<a href="https://colab.research.google.com/github/crislyn-bot/E2E-Transformer-Tboli-English-NMT/blob/main/asr_whisper_tboli_lang.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>



> This .ipynb contains the source code of the asr-whisper-large-v3-tboli-lang during the fine-tuning process. It was fine-tuned on openai/whisper-large-v3 adapted using Parameter-Efficient Fine-Tuning (PEFT) with Low-Rank Adaptation (LoRA) for Automatic Speech Recognition (ASR) of the low-resource Tboli language



# ASR (WHISPER)

In [ ]:
# Install Dependencies and Mount GDrive
!pip install --upgrade pip
!pip install --upgrade datasets[audio] transformers accelerate evaluate jiwer tensorboard peft gradio librosa pandas
!apt-get update && apt-get install -y ffmpeg

# IMPORTANT: After running this cell, please restart the runtime (Runtime > Restart runtime) and then run all cells from the beginning.

In [ ]:
# Install FFmpeg system dependencies (required for audio decoding)
!apt update -qq && apt install -y ffmpeg

# Uninstall any conflicting torchaudio/torchcodec (if present)
!pip uninstall -y torchaudio torchcodec

# Install compatible PyTorch and TorchCodec versions (e.g., torch 2.9.x with torchcodec 0.8.x)
!pip install "torch==2.9.*" "torchcodec==0.8.*"

# Verify installations (for checking)
import torch
import torchcodec
print(torch.__version__)
print(torchcodec.__version__)
!ffmpeg -version


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

# Ensure a clean copy of the dataset from Drive ---
!rm -rf /content/tboli_dataset  # Remove any previous incomplete or incorrect copy
!mkdir -p /content/tboli_dataset # Ensure the target directory exists
!cp -r /content/drive/MyDrive/tboli_dataset/* /content/tboli_dataset/ # Copy contents

In [ ]:
from huggingface_hub import logout; logout()

In [ ]:
# Hugging Face Login (for pushing model if desired)
from huggingface_hub import notebook_login
notebook_login()  # Paste your HF token


In [ ]:
# Load Metadata CSVs and Create Dataset
import pandas as pd
from datasets import Dataset, DatasetDict, Audio

# Load clean metadata (full with splits)
clean_df = pd.read_csv('/content/tboli_dataset/metadata_clean.csv')  # Adjust path

# Load augmented train metadata
aug_df = pd.read_csv('/content/tboli_dataset/metadata_aug_1.csv')  # Adjust path

# Combine train: clean train + augmented
train_df = pd.concat([clean_df[clean_df['split'] == 'train'], aug_df])

# Val and test from clean
val_df = clean_df[clean_df['split'] == 'val']
test_df = clean_df[clean_df['split'] == 'test']


In [ ]:
# Create DatasetDict (use 'tboli_script' as transcription labels)

audio_base_dir = "/content/tboli_dataset/"

def create_audio_path(row):
    speaker_id_prefix = row['id'].split('_')[0]
    if '_aug' in row['id']: # Check if it's an augmented sample
        return f"{audio_base_dir}augmented_audio/{speaker_id_prefix}/{row['id']}.wav"
    else: # It's a clean sample
        return f"{audio_base_dir}cleaned_audio/{speaker_id_prefix}/{row['id']}.wav"

# Apply the function to create 'audio_path' for each DataFrame
train_df['audio_path'] = train_df.apply(create_audio_path, axis=1)
val_df['audio_path'] = val_df.apply(create_audio_path, axis=1)
test_df['audio_path'] = test_df.apply(create_audio_path, axis=1)

# Create a new 'audio' column as a list of dictionaries, which is what the Audio feature expects internally
train_df['audio'] = [{'path': p} for p in train_df['audio_path']]
val_df['audio'] = [{'path': p} for p in val_df['audio_path']]
test_df['audio'] = [{'path': p} for p in test_df['audio_path']]

tboli_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df).cast_column('audio', Audio(sampling_rate=16000)),
    'validation': Dataset.from_pandas(val_df).cast_column('audio', Audio(sampling_rate=16000)),
    'test': Dataset.from_pandas(test_df).cast_column('audio', Audio(sampling_rate=16000))
})

In [ ]:
# Remove unnecessary columns
tboli_dataset = tboli_dataset.remove_columns(['script_id', 'id', 'english_script', 'speaker_id', 'split'])
tboli_dataset = tboli_dataset.rename_column('tboli_script', 'sentence')

In [ ]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

model_name = "openai/whisper-large-v3"

# proxy language (Tagalog) ---
tokenizer = WhisperTokenizer.from_pretrained(
    model_name,
    language="tl",  # Tagalog as proxy for Tboli
    task="transcribe"
)

processor = WhisperProcessor.from_pretrained(
    model_name,
    language="tl",  # Tagalog as proxy
    task="transcribe"
)

feature_extractor = processor.feature_extractor


In [ ]:
def prepare_dataset(batch, feature_extractor, tokenizer):  # Batched version
    audios = batch["audio"]
    # Extract features for list of arrays
    batch["input_features"] = feature_extractor(
        [a["array"] for a in audios],
        sampling_rate=audios[0]["sampling_rate"]
    ).input_features
    # Tokenize list of sentences
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

tboli_dataset = tboli_dataset.map(
    prepare_dataset,
    batched=True,
    batch_size=4,
    remove_columns=tboli_dataset["train"].column_names,
    fn_kwargs={"feature_extractor": feature_extractor, "tokenizer": tokenizer},
    num_proc=None  # Changed from 1 to None for single-process execution
)

In [ ]:
#Data Collator
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any  # Changed to field with type hint
    decoder_start_token_id: int  # Changed to field with type hint

    def __call__(self, features):
        # Audio features
        input_features = ([{"input_features": f["input_features"]} for f in features])
        batch = self.processor.feature_extractor.pad(
            input_features,
            return_tensors="pt"
        )

        batch["input_features"] = batch["input_features"].half()

        # Labels
        label_features = ([{"input_ids": f["labels"]} for f in features])
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            return_tensors="pt"
        )

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )

        if (labels[:, 0] == self.decoder_start_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor, decoder_start_token_id=tokenizer.pad_token_id)  # Adjust if needed

In [ ]:
# Load Model with PEFT/LoRA for Few-Shot Efficiency
from transformers import WhisperForConditionalGeneration
from peft import LoraConfig, get_peft_model

model = WhisperForConditionalGeneration.from_pretrained(model_name)
model.config.use_cache = False

model.generation_config.task = "transcribe"
model.generation_config.forced_decoder_ids = None
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id


In [ ]:
# For inspection of the encoder layer path
from transformers import WhisperForConditionalGeneration

#  Load model
model = WhisperForConditionalGeneration.from_pretrained(model_name)

# Inspect submodules
for name, module in model.named_modules():
    if "encoder" in name:
        print(name)

In [ ]:
# Freeze all encoder layers first
for name, param in model.base_model.encoder.named_parameters():
    param.requires_grad = False

# Unfreeze the last 2 encoder layers (30 and 31)
for name, param in model.base_model.encoder.named_parameters():
    if "layers.30" in name or "layers.31" in name:
        param.requires_grad = True

In [ ]:
from peft import LoraConfig, get_peft_model
# Apllly Lora
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # decoder attention modules
    lora_dropout=0.05
)

model = get_peft_model(model, lora_config)


In [ ]:
#Evaluation Metric (WER)
import evaluate

metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = tokenizer.pad_token_id
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)
    wer = 100 * metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}

In [ ]:
# Training Arguments (Optimized for Colab T4)
from transformers import Seq2SeqTrainingArguments, IntervalStrategy

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-tboli-asr-baseline",  # Save to GDrive if needed
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-6,
    warmup_steps=100,
    num_train_epochs=12,
    gradient_checkpointing=True,  # Enabled to save memory
    fp16=True,  # Mixed precision
    eval_strategy="steps",
    per_device_eval_batch_size=4,
    predict_with_generate=True,
    generation_max_length=225,  # For short phrases
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True  # Push to HF
)

In [ ]:
# Trainer
from transformers import Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=tboli_dataset["train"],
    eval_dataset=tboli_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Set tokenizer and feature_extractor after initialization (for generation, evaluation, and saving)
trainer.tokenizer = processor

In [ ]:
print(next(model.parameters()).dtype)  # torch.float16
print(model.config.use_cache)          # False
print(trainer.tokenizer.__class__)     # WhisperProcessor


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Train!
trainer.train()

In [ ]:
import evaluate

cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    # Decode
    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * metric.compute(
        predictions=pred_str,
        references=label_str
    )

    cer = 100 * cer_metric.compute(
        predictions=pred_str,
        references=label_str
    )

    return {
        "wer": wer,
        "cer": cer
    }

In [ ]:
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=tboli_dataset["train"],
    eval_dataset=tboli_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)


In [ ]:
trainer.evaluate()


In [ ]:
trainer.save_model("./whisper-tboli-asr-baseline/pre-asr")
processor.save_pretrained("./whisper-tboli-asr-baseline/pre-asr")

print("\n PRE-RLHF training complete!")

In [ ]:
trainer.push_to_hub()

In [ ]:
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from peft import PeftModel
import torch

peft_model_path = "./whisper-tboli-asr-baseline/pre-asr"

base_model = WhisperForConditionalGeneration.from_pretrained(
    "openai/whisper-large-v3",
    torch_dtype=torch.float16
).to("cuda")

model = PeftModel.from_pretrained(base_model, peft_model_path)
model.eval()

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-large-v3",
    language="tl",  # Tagalog as proxy for Tboli, as per earlier setup
    task="transcribe"
)


trainer.model = model
trainer.tokenizer = processor

eval_results = trainer.evaluate(tboli_dataset["validation"])

val_cer = eval_results.get("eval_cer", "CER not found in results")

print(f"Validation Evaluation Results: {eval_results}")
print(f"CER on validation set: {val_cer}")